# Part C1: Receptive Field, Dilated & Strided Convolutions

---

## The Receptive Field Problem

You now know that a Conv1D kernel slides over time and at each position looks at `kernel_size` samples. A kernel of size 5 looks at 5 samples — that's its **receptive field**: the number of input samples that influence one output value.

At 16kHz, 5 samples is 0.3ms. That's nothing — a single pitch period of speech is ~6ms (96 samples). A syllable is ~100ms (1600 samples). For your network to understand speech, it needs to "see" long stretches of audio at once.

So the obvious solution is: just use a huge kernel. Kernel size 1600, done?

**The problem with huge kernels:**

``` t
kernel_size=1600, 1 output channel:
Parameters = 1600 weights
But you want 256 channels:
Parameters = 1600 × 256 = 409,600 just for ONE layer
```

And even worse — a single huge kernel looks at everything at once with no hierarchy. It's like trying to understand a sentence by staring at all 20 words simultaneously instead of reading left to right.

The real question is: **how do you build a large receptive field efficiently?**

Two answers: **stacking layers** and **dilation**.

---

## Building Receptive Field by Stacking

If you stack multiple Conv1D layers, the receptive field grows. Each layer looks at `kernel_size` outputs from the previous layer, and each of those outputs already "saw" `kernel_size` inputs.

With kernel size 3, stride 1, no dilation:

```t
Layer 1: each output sees 3 inputs         → receptive field = 3
Layer 2: each output sees 3 layer-1 outputs 
         each of which saw 3 inputs         → receptive field = 5
Layer 3:                                    → receptive field = 7
Layer N:                                    → receptive field = 2N + 1
```

The formula for stacking `N` layers of kernel size `K`, stride 1:
```t
receptive_field = N × (K - 1) + 1
```

To get a receptive field of 1600 samples with kernel size 3:
```t
N = (1600 - 1) / (3 - 1) = 799 layers
```

799 layers. That's not feasible — too deep, too slow, vanishing gradients.

There has to be a better way.

---

## Dilated Convolutions: Skipping Samples

A **dilated convolution** (also called atrous convolution) introduces gaps between kernel elements. Instead of looking at consecutive samples, it looks at every `d`-th sample, where `d` is the **dilation rate**.

```t
Normal conv (dilation=1), kernel_size=3:
input:  [a, b, c, d, e, f, g]
kernel touches: [a, b, c]  ← 3 consecutive

Dilated conv (dilation=2), kernel_size=3:
kernel touches: [a, _, c, _, e]  ← skips every other sample
                 ↑    ↑    ↑
              pos 0  pos 2  pos 4

Dilated conv (dilation=4), kernel_size=3:
kernel touches: [a, _, _, _, e, _, _, _, i]
```

Same number of parameters (still just 3 weights), but the receptive field is now:
```t
receptive_field = dilation × (kernel_size - 1) + 1
```

For dilation=4, kernel_size=3: `4 × (3-1) + 1 = 9`

**The real power: exponential dilation stacking**

Conv-TasNet stacks layers with dilation rates `1, 2, 4, 8, 16, 32, 64, 128`. Each layer doubles the dilation. The total receptive field after 8 layers of kernel_size=3:

```t
Layer 1: dilation=1   → sees 3 samples
Layer 2: dilation=2   → sees 5 samples
Layer 3: dilation=4   → sees 9 samples
Layer 4: dilation=8   → sees 17 samples
Layer 5: dilation=16  → sees 33 samples
Layer 6: dilation=32  → sees 65 samples
Layer 7: dilation=64  → sees 129 samples
Layer 8: dilation=128 → sees 257 samples

Total receptive field = sum ≈ 510 samples = ~32ms
```

With 3 repetitions of this stack (what Conv-TasNet actually does): ~1.5 seconds of receptive field, using only 24 layers, each with kernel_size=3.

Compare: to get the same receptive field by stacking normal convolutions you'd need ~250 layers.

This is why dilated convolutions are central to Conv-TasNet.

---

## Strided Convolutions: Downsampling Built-In

You already know stride from Part A — it controls how far the window moves between steps. When stride > 1, the output is shorter than the input.

```t
Input:  [a, b, c, d, e, f, g, h]  (length 8)
kernel_size=3, stride=2:
  step 0: looks at [a,b,c] → one output
  step 1: looks at [c,d,e] → one output
  step 2: looks at [e,f,g] → one output
Output length = floor((8-3)/2) + 1 = 3
```

You've halved the time dimension. This is **downsampling** — compressing a long sequence into a shorter one.

**Why use strided conv instead of just pooling?**

Pooling (like `nn.MaxPool1d`) also downsamples, but it throws away information by taking max or average. A strided convolution *learns* how to downsample — the kernel weights are trained, so the network can learn to compress in a way that preserves what matters.

In Conv-TasNet's encoder: `Conv1d(kernel_size=16, stride=8)` takes 48,000 samples and outputs 6,000 feature vectors. That's the learned compression step before the TCN stack.

**Strided conv vs pooling:**

```t
MaxPool1d(stride=2): takes the maximum of every 2 values — fixed operation
Conv1d(stride=2):    learns a weighted combination — trainable
```

For audio, learned downsampling consistently outperforms fixed pooling because the network can learn to preserve frequency-relevant information.

---

## Putting It Together: Why These Three Concepts Connect

```t
Problem: need large receptive field
↓
Naive solution: big kernel or many layers → too expensive
↓
Solution 1: strided conv → compress time dimension first (encoder)
Solution 2: dilated conv → exponential receptive field growth (TCN stack)
↓
Result: Conv-TasNet architecture
  Encoder: Conv1d with large stride → 48000 → 6000 frames
  TCN: stacked dilated convs → huge receptive field cheaply  
  Decoder: (Part C2) reverse the encoding → 6000 → 48000
```

---

## TODOs

Two TODOs for C1 — one to build intuition for receptive fields, one to implement and verify dilated convolutions.

---

### TODO 1: Receptive Field Calculator

**Goal:** Build a function that computes the receptive field of a stack of Conv1D layers, and use it to compare different architectures.

**Requirements:**
- Write a function `receptive_field(layers)` where `layers` is a list of dicts, each with `kernel_size`, `stride`, and `dilation`
- The formula for how one layer contributes to receptive field growth is:

```t
RF_new = RF_old + (kernel_size - 1) × dilation × jump_old
jump_new = jump_old × stride
```

Start with `RF = 1`, `jump = 1`. Apply each layer in order.

- Use your function to compute and print the receptive field for these three architectures:
  - **Stack A:** 8 layers, kernel_size=3, stride=1, dilation=1 (no dilation, no stride)
  - **Stack B:** 8 layers, kernel_size=3, stride=1, dilation doubles each layer (1,2,4,8,16,32,64,128)
  - **Stack C:** Same as B but repeated 3 times (24 layers total — what Conv-TasNet uses)

**Expected output:** Three numbers in samples. Convert each to milliseconds (divide by 16000, multiply by 1000). Stack C should be well over 1 second.

**Key question:** How many layers would Stack A need to match Stack C's receptive field?

---

### TODO 2: Implement and Verify Dilated Conv1D

**Goal:** Use `torch.nn.Conv1d` with dilation and verify that the output shape matches the formula. Then verify that stacking two dilated layers gives the expected receptive field by testing with an impulse signal.

**Requirements:**

Part 1 — Shape verification:
- Create a `Conv1d` with `kernel_size=3`, `dilation=4`
- Run a `(1, 1, 48000)` input through it
- Compute the expected output length using the formula: `floor((L + 2p - d*(k-1) - 1) / s) + 1`
- Assert your formula matches the actual output shape

Part 2 — Impulse test (this is the interesting one):
- Create a signal of all zeros with a single `1.0` at position 500 — this is an **impulse**
- Pass it through a dilated Conv1d with `dilation=8, kernel_size=3`
- Find all the positions in the output that are non-zero (`torch.nonzero`)
- The non-zero positions tell you exactly which input positions influenced the output — that's the receptive field in action

**Hints:**
- `torch.zeros(1, 1, 1000)` then `signal[0, 0, 500] = 1.0`
- `torch.nn.Conv1d(1, 1, kernel_size=3, dilation=8, bias=False)`
- After forward pass, `torch.nonzero(output.squeeze())` gives you the positions
- With dilation=8, kernel_size=3: the kernel touches positions `[500-8, 500, 500+8]` in the input — where should non-zero outputs appear?

**Key question:** How many non-zero positions do you get in the output? Does it match `kernel_size`? Why?

In [501]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchaudio.transforms as T
import math
from pathlib import Path
import os
import sys
import json
import matplotlib.pyplot as plt

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


In [502]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        mixture_tensor = torch.from_numpy(mixture_audio)
        
        max_len = 48000
        
        if mixture_tensor.size(0) > max_len:
            # Truncate if too long
            mixture_tensor = mixture_tensor[:max_len]
        else:
            # Pad with zeros if too short
            padding = max_len - mixture_tensor.size(0)
            mixture_tensor = torch.nn.functional.pad(mixture_tensor, (0, padding))
        
        mixture_tensor = mixture_tensor.unsqueeze(0)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return mixture_tensor, label_tensor

In [503]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

10000
2000


In [504]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

### todo 1

In [505]:
def receptive_field(layers):
    rf = 1
    jump = 1
    for layer in layers:
        rf += (layer['kernel_size'] - 1) * layer['dilation'] * jump
        jump = jump * layer['stride']
    
    return rf

In [506]:
stack_a = [
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 1}
]

stack_b = [
    {"kernel_size": 3, "stride": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "dilation": 2},
    {"kernel_size": 3, "stride": 1, "dilation": 4},
    {"kernel_size": 3, "stride": 1, "dilation": 8},
    {"kernel_size": 3, "stride": 1, "dilation": 16},
    {"kernel_size": 3, "stride": 1, "dilation": 32},
    {"kernel_size": 3, "stride": 1, "dilation": 64},
    {"kernel_size": 3, "stride": 1, "dilation": 128}
]

# You can create this by repeating stack_b three times
stack_c = stack_b * 3

In [507]:
print(receptive_field(stack_a))
print(receptive_field(stack_b))
print(receptive_field(stack_c))

17
511
1531


### todo 2

In [508]:
audio = train_dataset[0][0].view(1,1,-1)

_, _, L = audio.shape
p = 0
s = 1
k = 3
d = 4
in_channels=1
out_channels=16

conv = nn.Conv1d(
    in_channels=in_channels, 
    out_channels=out_channels, 
    kernel_size=k, 
    stride=s, 
    padding=p, 
    dilation=d
    )

expected_length = math.floor((L + 2*p - d*(k - 1) - 1) / s) + 1

output = conv(audio)
actual_length = output.shape[2]

print(f"Formula predicts: {expected_length}")
print(f"Actual shape:    {actual_length}")

assert actual_length == expected_length, f"Error! Expected {expected_length} but got {actual_length}"

Formula predicts: 47992
Actual shape:    47992


In [509]:
signal = torch.zeros(1, 1, 1000)
signal[0, 0, 500] = 1.0

conv = nn.Conv1d(1, 1, kernel_size=3, dilation=8, bias=False)
output = conv(signal)

non_zero_indices = torch.nonzero(output.squeeze()).flatten()
items_list = non_zero_indices.tolist()
total_items = len(items_list)
counter = 0

print(f"Length of the list: {total_items}")
print(f"Non-zero outputs are at: {items_list}")

Length of the list: 3
Non-zero outputs are at: [484, 492, 500]


# Part C2: Depthwise-Separable Convolutions, Transposed Convolutions & Causal Padding

---

## Depthwise-Separable Convolutions: Doing More With Less

You know that a standard Conv1D with `in_channels=256, out_channels=256, kernel_size=3` has:

``` python
parameters = out_channels × in_channels × kernel_size
           = 256 × 256 × 3 = 196,608
```

Nearly 200k parameters for one layer. Conv-TasNet stacks dozens of these. That adds up fast.

Depthwise-separable convolution is a way to get nearly the same result with a fraction of the parameters. It splits one conv operation into two smaller ones:

### Step 1: Depthwise Convolution

Instead of mixing all input channels together, apply **one separate kernel per input channel**. Each channel gets its own filter, and channels never interact.

``` t
Standard conv:   each output channel = weighted sum across ALL input channels
Depthwise conv:  each output channel = ONE input channel filtered independently
```

So for `256 channels, kernel_size=3`:
``` t
Standard:   256 × 256 × 3 = 196,608 parameters
Depthwise:  1 × 256 × 3  =     768 parameters  (one kernel per channel)
```

In PyTorch, depthwise conv is just a regular `Conv1d` with `groups=in_channels`:

```python
Conv1d(256, 256, kernel_size=3, groups=256)
```

The `groups` parameter splits the channels into independent groups. When `groups == in_channels`, each channel is processed alone.

### Step 2: Pointwise Convolution

The depthwise step filtered each channel independently — but the channels still haven't talked to each other. Information from channel 5 hasn't influenced channel 200 at all.

The pointwise step fixes this: a `Conv1d` with `kernel_size=1` that mixes all channels at each time position.

```python
Conv1d(256, 256, kernel_size=1)
parameters = 256 × 256 × 1 = 65,536
```

### Combined Cost

``` t
Depthwise:   768 parameters
Pointwise:   65,536 parameters
Total:       66,304 parameters

vs Standard: 196,608 parameters

Savings: ~3× fewer parameters for similar expressiveness
```

### Why Does This Work?

The intuition: a standard conv does two things simultaneously — **spatial filtering** (looking at patterns over time) and **channel mixing** (combining information across channels). Depthwise-separable conv does these in sequence instead of simultaneously.

It turns out these two operations don't need to happen at the same time to get a good result. Separating them loses very little accuracy while cutting parameters significantly.

Conv-TasNet uses depthwise-separable convolutions in its TCN blocks precisely for this reason — it allows 256 channels with large receptive fields without the parameter count exploding.

---

## Transposed Convolutions: Running Conv Backwards

You've seen how a strided Conv1D compresses the time dimension:

``` t
Input:  (1, 256, 6000)   ← after encoder
Output: (1, 256, 750)    ← after strided conv with stride=8
```

But Conv-TasNet's decoder needs to go the other way — take a compressed representation and expand it back to the original length. This is what a **transposed convolution** does.

The name is confusing. It's not literally transposing a matrix — think of it as "the operation that inverts the shape change of a convolution."

Concretely, where a Conv1d with stride=2 halves the time dimension:

``` t
Conv1d(stride=2):     length 8 → length 4
ConvTranspose1d(stride=2): length 4 → length 8
```

### How It Works

A regular Conv1d with stride=2 slides a kernel, skipping every other position:

``` t 
Input positions:  0  1  2  3  4  5  6  7
                  ↓     ↓     ↓     ↓
Output positions: 0     1     2     3
```

A transposed conv with stride=2 does the reverse — each input position "broadcasts" its value into multiple output positions, with overlap summed:

``` t 
Input positions:  0     1     2     3
                  ↓     ↓     ↓     ↓
Output positions: 0  1  2  3  4  5  6  7
```

In PyTorch: `nn.ConvTranspose1d(in_channels, out_channels, kernel_size, stride)`

The output length formula for ConvTranspose1d:
```python
output_length = (input_length - 1) × stride - 2×padding + kernel_size
```

Notice this is the *inverse* of the Conv1d formula — stride now *multiplies* the length instead of dividing it.

### Where You'll Use This

In Conv-TasNet:
``` t 
Encoder: Conv1d(1, 512, kernel_size=16, stride=8)
         → (1, 1, 48000) becomes (1, 512, 5998)

Decoder: ConvTranspose1d(512, 1, kernel_size=16, stride=8)  
         → (1, 512, 5998) becomes (1, 1, 48000)
```

The decoder reconstructs the separated audio waveform from the compressed representation.

---

## Causal Padding: Not Cheating With the Future

Every convolution needs to handle boundaries — what happens when the kernel is at the edge of the sequence and needs samples that don't exist?

You saw "same" padding in Part A: pad zeros on both sides so the output length equals the input length.

But "same" padding pads **symmetrically** — it adds zeros on the left (past) AND right (future) side. For a kernel of size 3 centered at position 0:

``` t 
Same padding:   [0, x₀, x₁]   ← borrows one zero from the future
Causal padding: [0, 0,  x₀]   ← only borrows from the past
```

**Why does this matter?**

In a real-time system — like a phone call running speaker diarization live — at time `t` you only have access to audio up to `t`. You cannot look at `t+1` because it hasn't happened yet.

If your network uses same padding, it's secretly looking 1-2 samples into the future at every layer. Stack 10 layers and the network is peeking hundreds of milliseconds ahead. This is fine for offline processing, but breaks completely for real-time use.

**Causal padding** pads only on the left (past side), never the right:

``` t 
Same padding:    pad (kernel_size//2) on left AND right
Causal padding:  pad (kernel_size - 1) on left, 0 on right
```

PyTorch doesn't have a built-in "causal" padding option in Conv1d — you apply it manually before the conv:

```python
# For kernel_size=3, you need 2 samples of left padding
x = F.pad(x, (kernel_size - 1, 0))  # (left_pad, right_pad)
output = conv(x)  # conv with padding=0
```

The output length stays the same as the input length, but you've guaranteed the network never saw the future.

**EEND uses causal convolutions** because diarization needs to work in real-time streaming scenarios. Conv-TasNet can be either causal or non-causal depending on whether you need real-time operation.

---

## How These Three Fit Together in Conv-TasNet

``` t 
[Raw audio: (1, 1, 48000)]
        ↓
Encoder: Conv1d(stride=8)          ← strided conv (Part C1)
        ↓
[Encoded: (1, 512, 6000)]
        ↓
TCN Block × 24:
  - Depthwise-separable conv       ← efficient parameters
  - Dilated conv (Part C1)         ← large receptive field  
  - Causal padding                 ← no future leakage
        ↓
[Masks: (1, 512, 6000)]
        ↓
Decoder: ConvTranspose1d(stride=8) ← transposed conv
        ↓
[Separated audio: (1, 1, 48000)]
```

Every concept from C1 and C2 appears in this diagram. You now understand all of them.

---

## TODOs

Two TODOs. The first builds the parameter efficiency intuition concretely. The second verifies transposed conv as the inverse of conv, and implements causal padding correctly.

---

### TODO 1: Depthwise-Separable Conv — Parameter Count & Verification

**Goal:** Implement a depthwise-separable conv block, count its parameters, and verify it produces the same output shape as a standard conv.

**Requirements:**

Part 1 — Parameter counting (no code needed, just math):

Before writing anything, calculate on paper:
- Standard `Conv1d(in_channels=128, out_channels=128, kernel_size=5)`: how many parameters?
- Depthwise `Conv1d(128, 128, kernel_size=5, groups=128)`: how many parameters?
- Pointwise `Conv1d(128, 128, kernel_size=1)`: how many parameters?
- What is the ratio of standard to depthwise-separable?

Part 2 — Implementation:
- Build a `DepthwiseSeparableConv` as an `nn.Module` with two layers: a depthwise conv followed by a pointwise conv
- Both should have `in_channels=128, out_channels=128`
- Run a `(4, 128, 1000)` input through both your module and a standard `Conv1d(128, 128, kernel_size=5)`
- Verify the output shapes match
- Use `sum(p.numel() for p in model.parameters())` to count and print parameters for both

**Hints:**
- Depthwise conv: `groups=in_channels` in `Conv1d`
- Pointwise conv: `kernel_size=1`, no groups
- The depthwise kernel_size should be 5, the pointwise should be 1
- Output shapes will match but the values will differ — these are different operations with different weights, only the *shape* should be the same

**Key question:** Your depthwise conv needs padding to preserve sequence length. How much padding does a kernel_size=5, dilation=1 conv need on each side to keep the output the same length as the input?

---

### TODO 2: Transposed Conv + Causal Padding

**Goal:** Verify that ConvTranspose1d inverts Conv1d's shape change. Then implement causal padding and prove the network can't see the future.

**Requirements:**

Part 1 — Shape inversion:
- Create a `Conv1d(1, 16, kernel_size=16, stride=8)` — this is the Conv-TasNet encoder
- Run a `(1, 1, 48000)` input through it, print the output shape
- Create a `ConvTranspose1d` that maps back to `(1, 1, 48000)` — figure out the right parameters
- Verify the final shape matches the original input shape

Part 2 — Causal padding:
- Create a `Conv1d(1, 1, kernel_size=5, padding=0, bias=False)` with all weights set to `1.0`
- Create a signal: zeros everywhere except `signal[0, 0, 10] = 1.0`
- Apply causal padding manually using `F.pad`, then run through the conv
- Find the non-zero positions in the output
- **Verify:** the latest non-zero output position should be at index 10, never beyond it — the conv can only "see" the impulse and its past, not its future

**Hints:**
- For causal padding with kernel_size=5: you need `kernel_size - 1 = 4` zeros on the left, 0 on the right
- `F.pad(x, (4, 0))` — remember the format is `(left, right)`
- Setting all weights to 1.0: `conv.weight.data.fill_(1.0)`
- After the causal-padded conv, the non-zero outputs should span from index `10 - (kernel_size-1)` to index `10` — think about why

**Key question:** If you used same padding instead of causal padding here, what would the non-zero output positions be? How many samples into the future would the network peek?

---

Start with TODO 1. Show me the parameter counts from your pencil-and-paper calculation before you write any code.

### todo 1

part 1

Standard: - 128*128*5 = 81,920 
Depthwise: - 1*128*5 = 640 
Pointwise:- 128*128*1 = 16,384 
ratio = 81920/17024 =4.812 

part 2

In [510]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels=128, out_channels=128):
        super().__init__()
        
        self.Depthwise = nn.Conv1d(
            in_channels=in_channels, 
            out_channels=out_channels, 
            kernel_size=5, 
            groups=in_channels,
            bias=False,
            padding=2
            )
        self.Pointwise = nn.Conv1d(
            in_channels=in_channels, 
            out_channels=out_channels, 
            kernel_size=1, 
            bias=False,
            padding=2 
            ) 
        
    def forward(self, x):
        x = self.Depthwise(x)
        x = self.Pointwise(x)
        return x

In [511]:
sample = torch.randn([4, 128, 1000])

model = DepthwiseSeparableConv()

model.eval()
logits = model(sample)

print(sum(p.numel() for p in model.parameters()))

17024


### todo 2

In [512]:
sample = torch.randn([1, 1, 48000])

encoder = nn.Conv1d(1, 16, kernel_size=16, stride=8) 
decoder = nn.ConvTranspose1d(16, 1, kernel_size=16, stride=8)

encoded = encoder(sample)
decoded = decoder(encoded)

print(f"Input shape: {sample.shape}")
print(f"Encoded shape: {encoded.shape}")
print(f"Decoded shape: {decoded.shape}")

assert decoded.shape == sample.shape, f"Shape mismatch: {decoded.shape} != {sample.shape}"

Input shape: torch.Size([1, 1, 48000])
Encoded shape: torch.Size([1, 16, 5999])
Decoded shape: torch.Size([1, 1, 48000])


## Causal Padding — Clean Summary

**Setup:**
```t
signal (original):  [0,0,0,0,0,0,0,0,0,0, 1, 0,0,0,...]
index:               0 1 2 3 4 5 6 7 8 9  10 11 ...
                                           ↑ impulse
```

**After left-padding by 4 (kernel_size-1):**
```t
new_signal:  [0,0,0,0, 0,0,0,0,0,0,0,0,0,0, 1, 0,0,0,...]
index:        0 1 2 3  4 5 6 7 8 9 ...      14 15 ...
              ←padding→ ←── original signal shifted right by 4 ──→
```

**Conv with kernel_size=5, all weights=1.0, output[i] = sum of new_signal[i..i+4]:**

```t
output[10] = new_signal[10+11+12+13+14] = 0+0+0+0+1 = 1  ← first to see impulse
output[11] = new_signal[11+12+13+14+15] = 0+0+0+1+0 = 1
output[12] = new_signal[12+13+14+15+16] = 0+0+1+0+0 = 1
output[13] = new_signal[13+14+15+16+17] = 0+1+0+0+0 = 1
output[14] = new_signal[14+15+16+17+18] = 1+0+0+0+0 = 1  ← last to see impulse
output[15] = new_signal[15+16+17+18+19] = 0+0+0+0+0 = 0  ← impulse gone
```

**Non-zero output positions in padded coordinates: `[10, 11, 12, 13, 14]`**

**Subtract padding offset (4) to get original input coordinates:**
```t
[10-4, 11-4, 12-4, 13-4, 14-4] = [6, 7, 8, 9, 10]
```

**The causal guarantee:**
```t
Original impulse was at position 10.
Latest original position that influenced any output: 10.
Network never looked at position 11, 12, 13... → no future leakage. ✓
```

**What same padding would do instead:**
```t
Same padding: 2 on left, 2 on right
Non-zero outputs in original coordinates: [8, 9, 10, 11, 12]
                                                       ↑   ↑
                                              peeking 2 samples into the future ✗
```

---

The one-line summary: left-padding by `kernel_size-1` shifts the signal right so that when the kernel looks "forward" in the padded signal, it's actually looking at the past of the original signal.

Ready for Part D?

In [513]:
kernel_size = 5

signal = torch.zeros(1, 1, 1000)
signal[0, 0, 10] = 1.0

new_signal = F.pad(signal, (kernel_size-1, 0))
conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=0, bias=False)

with torch.no_grad():
    conv.weight.data.fill_(1.0)

output = conv(new_signal)
non_zero_indices = torch.nonzero(output.squeeze()).flatten()
items_list = non_zero_indices.tolist()
total_items = len(items_list)
counter = 0

print(f"Length of the list: {total_items}")
# Subtract the padding offset to get original input indices
original_positions = [i - (kernel_size - 1) for i in items_list]
print(f"Non-zero outputs correspond to original input positions: {original_positions}")


Length of the list: 5
Non-zero outputs correspond to original input positions: [6, 7, 8, 9, 10]
